# **IMPORT**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
N_INIT = 10

data_path = "dustinia_bersekolah.csv"
df = pd.read_csv(data_path)

display(df.head())

# **DATA TYPE, SHAPE, MISSING & DUPLICATE VALUES**

In [ ]:
df.info()

print(f"baris: {df.shape[0]}")
print(f"kolom: {df.shape[1]}")

print("Missing values pada tiap column:\n")
missing_ctr = df.isnull().sum()
print(missing_ctr[missing_ctr > 0])

dup_ctr = df.duplicated().sum()
print(f"duplicated record: {dup_ctr}")

# **DESCRIPTIVE**

In [ ]:
# deskripsi data
print("Statistik Deskriptif (Kolom Numerik)")
display(df.describe())

# **HEATMAP CORRELATION TO DECIDE FEATURE USED**

In [ ]:
numerical_features = [
    "age",
    "study_hours_per_week",
    "attendance_rate",
    "extracurricular_activities",
    "tutoring_sessions",
    "sleep_hours",
    "stress_level",
    "motivation_score",
    "reading_score",
    "writing_score",
    "math_score",
    "science_score",
    "overall_gpa",
]

corr_matrix = df[numerical_features].corr()

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True
)

plt.title("Correlation Heatmap of Numerical Features", fontsize=14, pad=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.show()

# **OUTLIERS AND DISTRIBUTIONS OF SELECTED FEATURES**

Disini, kelompok kami memilih 4 fitur berikut yakni,

1. overall_gpa
2. study_hours_per_week
3. sleep_hours
4. stress_level

Karena keempat fitur tersebut secara bersamaan menggambarkan proses (effort dan well-being) sekaligus hasil (capaian akademik) siswa. Dengan memasukkan GPA langsung kedalam clustering, kami dapat memetakan siswa berdasarkan kombinasi utuh antara pola belajar, kedisiplinan, dan performanya.

Jika dilihat dari correlation heatmap pun, keempat fitur masih memiliki korelasi antara satu sama lain meskipun tidak sekuat fitur numerikal tentang nilai akademik siswa.

In [ ]:
selected_features = [
    "overall_gpa",
    "study_hours_per_week",
    "sleep_hours",
    "stress_level",
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for idx, col in enumerate(selected_features):
    sns.histplot(df[col], kde=True, ax=axes[0, idx], color="steelblue")
    axes[0, idx].set_title(f"Distribusi {col}", fontsize=12)

    sns.boxplot(x=df[col], ax=axes[1, idx], color="salmon")
    axes[1, idx].set_title(f"Boxplot {col}", fontsize=12)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=df,
    x="study_hours_per_week",
    y="overall_gpa",
    ax=axes[0],
    alpha=0.5,
    color="teal",
)
axes[0].set_title("Study Hours vs Overall GPA", fontsize=12)

sns.scatterplot(
    data=df,
    x="sleep_hours",
    y="stress_level",
    ax=axes[1],
    alpha=0.5,
    color="crimson",
)
axes[1].set_title("Sleep Hours vs Stress Level", fontsize=12)

plt.tight_layout()
plt.show()

# **STANDARDIZATION OF SELECTED FEATURE**

In [ ]:
X = df[selected_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=selected_features)
print("Data setelah standarisasi (5 baris teratas):")
display(df_scaled.head())

# **DECIDING K WITH ELBOW & SILHOUETTE METHODE**

In [ ]:
k_range = range(2, 11)
inertia_list = []
silhouette_list = []

for k in k_range:
    kmeans_eval = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    kmeans_eval.fit(X_scaled)

    inertia_list.append(kmeans_eval.inertia_)
    score = silhouette_score(X_scaled, kmeans_eval.labels_)
    silhouette_list.append(score)
    print(f"K = {k} -> Inertia: {kmeans_eval.inertia_:.2f}, Silhouette Score: {score:.4f}")

fig, ax1 = plt.subplots(figsize=(10, 5))

color = "tab:blue"
ax1.set_xlabel("Jumlah Cluster (K)")
ax1.set_ylabel("Inertia (Elbow Method)", color=color)
ax1.plot(k_range, inertia_list, marker="o", color=color, linewidth=2)
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
color = "tab:red"
ax2.set_ylabel("Silhouette Score", color=color)
ax2.plot(k_range, silhouette_list, marker="s", color=color, linestyle="--", linewidth=2)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Evaluasi K-Means: Elbow Method & Silhouette Score", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# **CLUSTERING BASED ON K**

In [ ]:
FINAL_K = 3

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=N_INIT)
cluster_labels = kmeans_final.fit_predict(X_scaled)

df["cluster"] = cluster_labels
print("Distribusi jumlah siswa per cluster:")
print(df["cluster"].value_counts().sort_index())

# **CLUSTER VISUALIZATION**

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(X_scaled)

df["pca_1"] = pca_result[:, 0]
df["pca_2"] = pca_result[:, 1]

var_ratio = pca.explained_variance_ratio_.sum() * 100

centroids_pca = pca.transform(kmeans_final.cluster_centers_)

plt.figure(figsize=(10, 7))

sns.scatterplot(
  data=df,
  x="pca_1",
  y="pca_2",
  hue="cluster",
  palette="Set2",
  alpha=0.7,
  s=40,
)

plt.scatter(
  centroids_pca[:, 0],
  centroids_pca[:, 1],
  s=250,
  c="black",
  marker="X",
  edgecolor="white",
  linewidths=1.5,
  label="Centroids",
  zorder=5,
)

plt.title(
  f"Visualisasi Cluster pada Dua Komponen PCA ({var_ratio:.1f}% variasi)",
  fontsize=13,
  pad=12,
)
plt.xlabel("PCA 1", fontsize=11)
plt.ylabel("PCA 2", fontsize=11)
plt.legend(
  bbox_to_anchor=(1.05, 1),
  loc="upper left",
  title="Cluster / Centroid",
)
plt.tight_layout()
plt.show()

# **PROFILING CLUSTERS**

In [ ]:
cluster_profile = df.groupby("cluster")[selected_features].mean().round(2)
display(cluster_profile)

cluster_name_mapping = {
    0: "High Achievers",
    1: "Stressed Strugglers",
    2: "Low-Effort",
}

df["cluster_name"] = df["cluster"].map(cluster_name_mapping)

print("Profil akhir tiap cluster:")
display(df.groupby(["cluster", "cluster_name"])[selected_features].mean().round(2))

In [ ]:
profile = df.groupby("cluster")[selected_features].mean()

profile_z = (profile - df[selected_features].mean()) / df[selected_features].std()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    profile_z,
    annot=profile.round(2),
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Z-score (relatif thd rata-rata keseluruhan)"},
    ax=axes[0],
)
axes[0].set_title(
    "Heatmap Profil Cluster\n(angka = nilai asli rata-rata, warna = z-score)",
    fontsize=12,
    pad=10,
)
axes[0].set_ylabel("Cluster", fontsize=11)
axes[0].set_yticklabels(
    [cluster_name_mapping[i] for i in profile.index],
    rotation=0,
)

profile_z_reset = profile_z.reset_index().melt(
    id_vars="cluster",
    var_name="fitur",
    value_name="z_score",
)
profile_z_reset["cluster_name"] = profile_z_reset["cluster"].map(cluster_name_mapping)

sns.barplot(
    data=profile_z_reset,
    x="fitur",
    y="z_score",
    hue="cluster_name",
    palette="Set2",
    ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title(
    "Perbandingan Profil Cluster (Standardized)",
    fontsize=12,
    pad=10,
)
axes[1].set_xlabel("Fitur", fontsize=11)
axes[1].set_ylabel("Z-score", fontsize=11)
axes[1].legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

# **EXPORT**

In [ ]:
output_csv_path = "dustinia_bersekolah_clustered.csv"
df.to_csv(output_csv_path, index=False)
print(f"Dataframe berhasil diexport ke {output_csv_path}")